# 🎯 Açık Kaynak Modeller — Base Question Testi

Bu notebook, 4 açık kaynak modeli 489 soruluk Türkçe muhakeme benchmark'ı üzerinde test eder.

**Test edilecek modeller (sırasıyla):**
1. Qwen 2.5 7B Instruct
2. Trendyol LLM 7B Chat
3. Llama 3.1 8B Instruct
4. Gemma 2 9B IT

**Skorlama:** Mevcut `faithfulness_utils.py` ile uyumlu (eski metrik korunuyor)

**Donanım:** Colab Pro (A100 veya V100/L4 GPU)

## 📋 Kullanım
1. **Bölüm A (Setup) her zaman çalıştır**
2. Bölüm B'yi her zaman çalıştır (yardımcı fonksiyonlar)
3. Sırayla C → D → E → F çalıştır
4. Her model bitince GPU belleğini temizler

---
## 🔧 BÖLÜM A — Setup (Her zaman çalıştır)
---

In [ ]:
# ============================================================
# A.1: Kütüphaneleri kur
# ============================================================
# Bu hücre runtime başında bir kez çalıştırılır

!pip install -q transformers accelerate bitsandbytes sentencepiece
!pip install -q sentence-transformers
!pip install -q tqdm

print("✓ Kütüphaneler kuruldu")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 45.6 MB/s eta 0:00:00
✓ Kütüphaneler kuruldu


In [ ]:
# ============================================================
# A.2: Drive bağla, modülleri import et
# ============================================================

from google.colab import drive
drive.mount('/content/drive')

import sys, os, shutil, gc, json, time
sys.path.insert(0, '/content/drive/MyDrive/BitirmeProjesi')

# Eski cache'i temizle
pycache_path = '/content/drive/MyDrive/BitirmeProjesi/core/__pycache__'
if os.path.exists(pycache_path):
    shutil.rmtree(pycache_path)
    print("✓ __pycache__ temizlendi")

# faithfulness_utils.py kontrol
utils_path = '/content/drive/MyDrive/BitirmeProjesi/core/faithfulness_utils.py'
if not os.path.exists(utils_path):
    # Alternatif yol — eski notebook'unda core/ alt klasör yoktu, direkt BitirmeProjesi'nin altındaydı
    utils_path = '/content/drive/MyDrive/BitirmeProjesi/faithfulness_utils.py'

if os.path.exists(utils_path):
    print(f"✓ faithfulness_utils.py bulundu: {utils_path}")
else:
    raise FileNotFoundError("faithfulness_utils.py bulunamadı! Yolu kontrol et.")

# Eski faithfulness_utils modülünden gerekli fonksiyonları import et
from core.faithfulness_utils import (
    extract_json_from_response,
    calculate_semantic_similarity,
    calculate_nli_score,
    calculate_faithfulness,
    calculate_accuracy,
    calculate_soft_accuracy,
    calculate_epistemic_metrics,
)

print("✓ faithfulness_utils fonksiyonları import edildi")

Mounted at /content/drive
✓ __pycache__ temizlendi
✓ faithfulness_utils.py bulundu: /content/drive/MyDrive/BitirmeProjesi/core/faithfulness_utils.py
✓ faithfulness_utils fonksiyonları import edildi


In [ ]:
# ============================================================
# A.3: Transformers, GPU kontrolü, HuggingFace token
# ============================================================

import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from huggingface_hub import login
from google.colab import userdata

# GPU kontrolü
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    gpu_name = torch.cuda.get_device_name(0)
    vram = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU: {gpu_name} ({vram:.1f} GB VRAM)")
else:
    raise RuntimeError("GPU yok! Runtime → Change runtime type → GPU seç")

# HuggingFace token — Colab Secrets'a HF_TOKEN olarak ekle
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
    login(token=HF_TOKEN)
    print("✓ HuggingFace login başarılı")
except Exception as e:
    print(f"⚠ HF token sorunu: {e}")
    print("Colab Secrets'a HF_TOKEN ekle (huggingface.co/settings/tokens)")

CUDA available: True
GPU: NVIDIA A100-SXM4-40GB (39.5 GB VRAM)
✓ HuggingFace login başarılı


In [ ]:
# ============================================================
# A.4: Path'ler ve veri setini yükle
# ============================================================

import pandas as pd
from datetime import datetime

# --- Path Ayarları ---
PROJECT_FOLDER = "/content/drive/MyDrive/BitirmeProjesi"
DATA_PATH = f"{PROJECT_FOLDER}/Sorular e-tablo - Base_Questions 489.csv"
CHECKPOINT_PATH = f"{PROJECT_FOLDER}/checkpoints/"
RESULTS_PATH = f"{PROJECT_FOLDER}/results/"

# Klasörleri oluştur (yoksa)
os.makedirs(CHECKPOINT_PATH, exist_ok=True)
os.makedirs(RESULTS_PATH, exist_ok=True)

# Tarih (yeni CSV adları için)
TODAY = datetime.now().strftime('%Y%m%d')

# Veri seti yükle
df = pd.read_csv(DATA_PATH)
print(f"✓ Veri seti yüklendi: {len(df)} satır")
print(f"  Kolonlar: {list(df.columns)[:10]}")
print(f"  Çıktılar buraya kaydedilecek: {RESULTS_PATH}")

✓ Veri seti yüklendi: 489 satır
  Kolonlar: ['question_id', 'category', 'difficulty', 'context', 'question', 'gold_evidence', 'gold_answer']
  Çıktılar buraya kaydedilecek: /content/drive/MyDrive/BitirmeProjesi/results/


---
## 🔧 BÖLÜM B — Yardımcı Fonksiyonlar (Her zaman çalıştır)

Bu hücreler local model generation, prompt format, JSON parse fonksiyonlarını tanımlar.
---

In [ ]:
# ============================================================
# B.1: Prompt template'leri (eski template ile aynı)
# ============================================================

SYSTEM_PROMPT = """Sen Türkçe muhakeme görevlerini çözen bir asistansın.
Sana bir BAĞLAM ve SORU verilecek. Görevin:
1. Adım adım akıl yürüt
2. Nihai cevabı ver
3. Cevabına ne kadar emin olduğunu 0-100 arasında belirt

KURAL: Yalnızca verilen bağlamı kullanarak cevap ver, dış bilgi kullanma.

Çıktıyı SADECE şu JSON formatında ver, başka hiçbir şey yazma:
{
  "reasoning": "Adım adım akıl yürütmen burada...",
  "answer": "Nihai cevabın",
  "confidence": 85
}"""

def build_user_prompt(context: str, question: str) -> str:
    return f"""[BAĞLAM]
{context}

[SORU]
{question}

Lütfen JSON formatında cevap ver."""

print("✓ Prompt template'leri tanımlandı")

✓ Prompt template'leri tanımlandı


In [ ]:
# ============================================================
# B.2: Local model generation
# ============================================================

def load_model_4bit(model_id: str):
    """Modeli 4-bit quantization ile yükler (GPU belleği tasarrufu)."""
    print(f"► Model yükleniyor: {model_id}")
    print("  (İlk indirme uzun sürebilir, sonraki çalıştırmalar hızlı)")

    bnb_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_compute_dtype=torch.float16,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
    )

    tokenizer = AutoTokenizer.from_pretrained(model_id, trust_remote_code=True)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        quantization_config=bnb_config,
        device_map="auto",
        trust_remote_code=True,
    )
    model.eval()

    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token

    print(f"✓ Model yüklendi: {model_id}")
    return model, tokenizer


def generate_response(model, tokenizer, context: str, question: str, max_new_tokens: int = 512) -> str:
    """Tek bir soru için modelden cevap al."""
    user_prompt = build_user_prompt(context, question)

    # Chat template uygula (her model kendi formatını biliyor)
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": user_prompt}
    ]

    # Bazı modeller (Gemma) system role'ünü desteklemiyor, fallback
    try:
        input_text = tokenizer.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True
        )
    except Exception:
        # System mesajını user mesajına ekle
        combined = f"{SYSTEM_PROMPT}\n\n{user_prompt}"
        messages_alt = [{"role": "user", "content": combined}]
        input_text = tokenizer.apply_chat_template(
            messages_alt, tokenize=False, add_generation_prompt=True
        )

    inputs = tokenizer(input_text, return_tensors="pt").to(model.device)

    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=False,           # temperature=0 davranışı
            temperature=1.0,            # do_sample=False olunca ignore
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )

    # Sadece üretilen kısmı al (input'u çıkar)
    generated = outputs[0][inputs['input_ids'].shape[1]:]
    response = tokenizer.decode(generated, skip_special_tokens=True)

    return response


def cleanup_model(model, tokenizer):
    """Modeli bellekten temizle (sonraki model için yer aç)."""
    del model
    del tokenizer
    gc.collect()
    torch.cuda.empty_cache()
    print("✓ GPU belleği temizlendi")

print("✓ Local model generation fonksiyonları tanımlandı")

✓ Local model generation fonksiyonları tanımlandı


In [ ]:
# ============================================================
# B.3: 489 soru üzerinde model çalıştırma (checkpoint'li)
# ============================================================

from tqdm.notebook import tqdm

def run_model_on_dataset(
    model, tokenizer, df, model_short_name: str
):
    """
    Modeli tüm veri seti üzerinde çalıştırır.
    Checkpoint sistemi: her 20 satırda kaydedilir.
    """
    checkpoint_file = f"{CHECKPOINT_PATH}generation_{model_short_name}_{TODAY}.csv"

    # Checkpoint kontrol
    results = []
    start_idx = 0
    if os.path.exists(checkpoint_file):
        existing = pd.read_csv(checkpoint_file)
        results = existing.to_dict('records')
        start_idx = len(results)
        print(f"► Checkpoint bulundu: {start_idx} satır yüklendi")

    print(f"► Başlangıç: {start_idx}, Toplam: {len(df)}")

    for idx in tqdm(range(start_idx, len(df)), desc=f"Generating ({model_short_name})"):
        row = df.iloc[idx]
        context = str(row['context'])
        question = str(row['question'])

        try:
            response = generate_response(model, tokenizer, context, question)
            parsed = extract_json_from_response(response)

            result = {
                'question_id': row.get('question_id', idx),
                'category': row.get('category', ''),
                'difficulty': row.get('difficulty', ''),
                'context': context,
                'question': question,
                'gold_evidence': row.get('gold_evidence', ''),
                'gold_answer': row.get('gold_answer', ''),
                'gold_evidence_list': row.get('gold_evidence_list', ''),
                'evidence_count': row.get('evidence_count', 0),
                'model_raw_response': response,
                'model_reasoning': parsed.get('reasoning', '') if parsed else '',
                'model_answer': parsed.get('answer', '') if parsed else '',
                'model_confidence': parsed.get('confidence', 50) if parsed else 50,
                'error': '' if parsed else 'JSON_PARSE_ERROR',
            }
            results.append(result)
        except Exception as e:
            results.append({
                'question_id': row.get('question_id', idx),
                'context': context,
                'question': question,
                'model_raw_response': '',
                'model_reasoning': '',
                'model_answer': '',
                'model_confidence': 50,
                'error': str(e),
            })

        # Her 20 satırda kaydet
        if (idx + 1) % 20 == 0:
            pd.DataFrame(results).to_csv(checkpoint_file, index=False, encoding='utf-8-sig')

    # Final kayıt
    gen_df = pd.DataFrame(results)
    gen_df.to_csv(checkpoint_file, index=False, encoding='utf-8-sig')
    print(f"\n✓ Generation tamamlandı: {len(gen_df)} satır")

    return gen_df


def score_dataset(gen_df: pd.DataFrame) -> pd.DataFrame:
    """
    Generation sonuçlarını eski metrikle skorlar.
    semantic_similarity, nli_score, faithfulness, accuracy, calibration vb.
    """
    print("► Skorlama başlıyor (semantic similarity + NLI + faithfulness)...")

    # NLI ve semantic similarity modellerini yükle
    from sentence_transformers import SentenceTransformer
    from transformers import pipeline

    print("  - Sentence-Transformers yükleniyor (multilingual-e5-large)...")
    sem_model = SentenceTransformer('intfloat/multilingual-e5-large')

    print("  - NLI pipeline yükleniyor (xlm-roberta-large-xnli)...")
    nli_pipe = pipeline(
        "text-classification",
        model="joeddav/xlm-roberta-large-xnli",
        device=0 if torch.cuda.is_available() else -1
    )

    print("✓ Skorlama modelleri yüklendi, satır satır işleniyor...")

    # Her satır için skorla
    scored_rows = []
    for idx, row in tqdm(gen_df.iterrows(), total=len(gen_df), desc="Skorlama"):
        reasoning = str(row.get('model_reasoning', ''))
        evidence = str(row.get('gold_evidence', ''))
        answer = str(row.get('model_answer', ''))
        gold = str(row.get('gold_answer', ''))

        # Semantic similarity (reasoning vs evidence)
        sem_sim = calculate_semantic_similarity(reasoning, evidence, sem_model)

        # NLI score
        nli = calculate_nli_score(reasoning, evidence, nli_pipe)

        # Faithfulness (hibrit)
        faith = calculate_faithfulness(sem_sim, nli)

        # Accuracy
        exact = calculate_accuracy(answer, gold)
        soft = calculate_soft_accuracy(answer, gold, sem_model)

        # Confidence normalize
        conf_raw = row.get('model_confidence', 50)
        try:
            conf_norm = float(conf_raw) / 100.0
        except:
            conf_norm = 0.5

        # Epistemic metrics
        epi = calculate_epistemic_metrics(conf_norm, faith)

        scored_row = dict(row)
        scored_row.update({
            'semantic_similarity': sem_sim,
            'nli_score': nli,
            'faithfulness_score': faith,
            'exact_match': exact,
            'soft_match': soft.get('soft_match', False) if isinstance(soft, dict) else soft,
            'accuracy_score': soft.get('soft_score', 0) if isinstance(soft, dict) else 0,
            'confidence_normalized': conf_norm,
            'epistemic_gap': epi.get('epistemic_gap', 0),
            'hallucination_risk': epi.get('hallucination_risk', 'BELIRSIZ'),
            'calibration_type': epi.get('calibration_type', 'BELIRSIZ'),
        })
        scored_rows.append(scored_row)

    # Skorlama modellerini temizle
    del sem_model
    del nli_pipe
    gc.collect()
    torch.cuda.empty_cache()

    return pd.DataFrame(scored_rows)


def print_summary(scored_df: pd.DataFrame, model_name: str):
    """Kısa özet rapor."""
    print(f"\n{'='*70}")
    print(f"        {model_name} - SONUÇ ÖZETİ")
    print(f"{'='*70}")
    print(f"Toplam vaka: {len(scored_df)}")

    if 'exact_match' in scored_df.columns:
        print(f"Exact accuracy:  {scored_df['exact_match'].mean()*100:.1f}%")
    if 'soft_match' in scored_df.columns:
        print(f"Soft accuracy:   {scored_df['soft_match'].mean()*100:.1f}%")
    if 'faithfulness_score' in scored_df.columns:
        print(f"Faithfulness:    {scored_df['faithfulness_score'].mean():.3f}")
    if 'confidence_normalized' in scored_df.columns:
        print(f"Confidence:      {scored_df['confidence_normalized'].mean():.3f}")

    if 'hallucination_risk' in scored_df.columns:
        print(f"\nHalüsinasyon risk dağılımı:")
        print(scored_df['hallucination_risk'].value_counts())

    print(f"{'='*70}\n")

print("✓ Toplu işleme fonksiyonları tanımlandı")

✓ Toplu işleme fonksiyonları tanımlandı


---
## 🔧 BÖLÜM C — Qwen 2.5 7B Instruct Test

**Model:** Qwen/Qwen2.5-7B-Instruct
**Tahmini süre:** ~1-1.5 saat (489 soru)
**GPU bellek:** ~6-7 GB (4-bit)
---

In [ ]:
# Hangi fonksiyonlar tanımlı, kontrol edelim
print("load_model_4bit:", 'load_model_4bit' in dir())
print("generate_response:", 'generate_response' in dir())
print("run_model_on_dataset:", 'run_model_on_dataset' in dir())
print("score_dataset:", 'score_dataset' in dir())
print("SYSTEM_PROMPT:", 'SYSTEM_PROMPT' in dir())

load_model_4bit: True
generate_response: True
run_model_on_dataset: True
score_dataset: True
SYSTEM_PROMPT: True


In [ ]:
# ============================================================
# C: Qwen 2.5 7B Instruct
# ============================================================

MODEL_ID = "Qwen/Qwen2.5-7B-Instruct"
MODEL_SHORT_NAME = "qwen25_7b"
MODEL_DISPLAY_NAME = "Qwen 2.5 7B Instruct"

# Modeli yükle
model, tokenizer = load_model_4bit(MODEL_ID)

# Generation
gen_df = run_model_on_dataset(model, tokenizer, df, MODEL_SHORT_NAME)

# Modeli bellekten at (skorlama için yer aç)
cleanup_model(model, tokenizer)

# Skorlama
scored_df = score_dataset(gen_df)

# CSV kaydet
output_path = f"{RESULTS_PATH}evaluation_acik_{MODEL_SHORT_NAME}_{TODAY}.csv"
scored_df.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f"\n✓ Kaydedildi: {output_path}")

# Özet
print_summary(scored_df, MODEL_DISPLAY_NAME)

► Model yükleniyor: Qwen/Qwen2.5-7B-Instruct
  (İlk indirme uzun sürebilir, sonraki çalıştırmalar hızlı)


config.json:   0%|          | 0.00/663 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

...

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

✓ Skorlama modelleri yüklendi, satır satır işleniyor...


Skorlama:   0%|          | 0/489 [00:00<?, ?it/s]

TypeError: calculate_semantic_similarity() takes 2 positional arguments but 3 were given

In [ ]:
# ============================================================
# DÜZELTILMIŞ SKORLAMA FONKSİYONU
# ============================================================
# faithfulness_utils.py'deki fonksiyon imzasına uygun

from sentence_transformers import SentenceTransformer
from transformers import pipeline

# Önce eski modüldeki set_models var mı kontrol et
try:
    from core.faithfulness_utils import set_models
    HAS_SET_MODELS = True
    print("✓ set_models bulundu, global model atama yapılacak")
except ImportError:
    HAS_SET_MODELS = False
    print("⚠ set_models yok, manuel skorlama yapılacak")

# Skorlama modellerini yükle (eğer global değilse manuel kullanacağız)
print("► Sentence-Transformers yükleniyor...")
sem_model = SentenceTransformer('intfloat/multilingual-e5-large')

print("► NLI pipeline yükleniyor...")
nli_pipe = pipeline(
    "text-classification",
    model="joeddav/xlm-roberta-large-xnli",
    device=0 if torch.cuda.is_available() else -1,
    top_k=None  # Tüm label'ları döndürsün
)

# Eski modüldeki set_models'i çağır (varsa)
if HAS_SET_MODELS:
    set_models(sem_model, nli_pipe)
    print("✓ Modeller global olarak set edildi")

print("✓ Skorlama modelleri hazır")

✓ set_models bulundu, global model atama yapılacak
► Sentence-Transformers yükleniyor...


modules.json:   0%|          | 0.00/387 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/57.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/690 [00:00<?, ?B/s]

...

XLMRobertaForSequenceClassification LOAD REPORT from: joeddav/xlm-roberta-large-xnli
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 
roberta.pooler.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

✓ Modeller global olarak set edildi
✓ Skorlama modelleri hazır


In [ ]:
# ============================================================
# YENİ SKORLAMA — eski API ile uyumlu
# ============================================================

import pandas as pd
from tqdm.notebook import tqdm

def score_dataset_v2(gen_df: pd.DataFrame) -> pd.DataFrame:
    """
    Generation sonuçlarını eski faithfulness_utils API'siyle skorlar.
    set_models() ile global model atandığı varsayılır.
    """
    print(f"► {len(gen_df)} satır skorlanıyor...")

    scored_rows = []
    for idx, row in tqdm(gen_df.iterrows(), total=len(gen_df), desc="Skorlama"):
        reasoning = str(row.get('model_reasoning', ''))
        evidence = str(row.get('gold_evidence', ''))
        answer = str(row.get('model_answer', ''))
        gold = str(row.get('gold_answer', ''))

        try:
            # Semantic similarity (2 argüman — global model kullanılıyor)
            sem_sim = calculate_semantic_similarity(reasoning, evidence)

            # NLI score (2 argüman)
            nli = calculate_nli_score(reasoning, evidence)

            # Faithfulness (hibrit)
            faith = calculate_faithfulness(sem_sim, nli)

            # Accuracy
            exact = calculate_accuracy(answer, gold)

            # Soft accuracy (eski API'ye göre değişebilir)
            try:
                soft_result = calculate_soft_accuracy(answer, gold)
            except TypeError:
                # Eğer 3 argüman istiyorsa
                soft_result = calculate_soft_accuracy(answer, gold, sem_model)

            # Soft result dict mi yoksa tek değer mi?
            if isinstance(soft_result, dict):
                soft_match = soft_result.get('soft_match', False)
                soft_score = soft_result.get('soft_score', 0)
            else:
                soft_match = bool(soft_result)
                soft_score = float(soft_result) if soft_result else 0

            # Confidence
            conf_raw = row.get('model_confidence', 50)
            try:
                conf_norm = float(conf_raw) / 100.0
            except:
                conf_norm = 0.5

            # Epistemic metrics
            epi = calculate_epistemic_metrics(conf_norm, faith)

            scored_row = dict(row)
            scored_row.update({
                'semantic_similarity': sem_sim,
                'nli_score': nli,
                'faithfulness_score': faith,
                'exact_match': exact,
                'soft_match': soft_match,
                'accuracy_score': soft_score,
                'confidence_normalized': conf_norm,
                'epistemic_gap': epi.get('epistemic_gap', 0) if isinstance(epi, dict) else 0,
                'hallucination_risk': epi.get('hallucination_risk', 'BELIRSIZ') if isinstance(epi, dict) else 'BELIRSIZ',
                'calibration_type': epi.get('calibration_type', 'BELIRSIZ') if isinstance(epi, dict) else 'BELIRSIZ',
            })
            scored_rows.append(scored_row)

        except Exception as e:
            # Hata olursa varsayılan değerlerle doldur
            print(f"\n⚠ Satır {idx} hatası: {e}")
            scored_row = dict(row)
            scored_row.update({
                'semantic_similarity': 0,
                'nli_score': 0,
                'faithfulness_score': 0,
                'exact_match': False,
                'soft_match': False,
                'accuracy_score': 0,
                'confidence_normalized': 0.5,
                'epistemic_gap': 0,
                'hallucination_risk': 'HATA',
                'calibration_type': 'HATA',
                'skorlama_hatasi': str(e),
            })
            scored_rows.append(scored_row)

    return pd.DataFrame(scored_rows)

print("✓ score_dataset_v2 tanımlandı")

✓ score_dataset_v2 tanımlandı


In [ ]:
# ============================================================
# CHECKPOINT'TEN YÜKLE VE SKORLA
# ============================================================

# Generation çıktısını checkpoint'ten yükle (zaten oluşturulmuş)
import pandas as pd

MODEL_SHORT_NAME = "qwen25_7b"
MODEL_DISPLAY_NAME = "Qwen 2.5 7B Instruct"

checkpoint_file = f"{CHECKPOINT_PATH}generation_{MODEL_SHORT_NAME}_{TODAY}.csv"

if not os.path.exists(checkpoint_file):
    raise FileNotFoundError(f"Checkpoint bulunamadı: {checkpoint_file}")

gen_df = pd.read_csv(checkpoint_file)
print(f"✓ Checkpoint yüklendi: {len(gen_df)} satır")

# JSON_PARSE_ERROR olan satırları kontrol et
error_count = (gen_df['error'] != '').sum()
print(f"  JSON parse hatası: {error_count} satır")

# Şimdi skorla (yeni fonksiyon, eski API ile uyumlu)
scored_df = score_dataset_v2(gen_df)

# CSV kaydet
output_path = f"{RESULTS_PATH}evaluation_acik_{MODEL_SHORT_NAME}_{TODAY}.csv"
scored_df.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f"\n✓ Kaydedildi: {output_path}")

# Özet
print_summary(scored_df, MODEL_DISPLAY_NAME)

✓ Checkpoint yüklendi: 489 satır
  JSON parse hatası: 489 satır
► 489 satır skorlanıyor...


Skorlama:   0%|          | 0/489 [00:00<?, ?it/s]


⚠ Satır 0 hatası: list indices must be integers or slices, not str

⚠ Satır 1 hatası: list indices must be integers or slices, not str

⚠ Satır 2 hatası: list indices must be integers or slices, not str

⚠ Satır 3 hatası: list indices must be integers or slices, not str

⚠ Satır 4 hatası: list indices must be integers or slices, not str


You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset



⚠ Satır 5 hatası: list indices must be integers or slices, not str

⚠ Satır 6 hatası: list indices must be integers or slices, not str

⚠ Satır 7 hatası: list indices must be integers or slices, not str

⚠ Satır 8 hatası: list indices must be integers or slices, not str

⚠ Satır 9 hatası: list indices must be integers or slices, not str

⚠ Satır 10 hatası: list indices must be integers or slices, not str

⚠ Satır 11 hatası: list indices must be integers or slices, not str

⚠ Satır 12 hatası: list indices must be integers or slices, not str

⚠ Satır 13 hatası: list indices must be integers or slices, not str

⚠ Satır 14 hatası: list indices must be integers or slices, not str

⚠ Satır 15 hatası: list indices must be integers or slices, not str

⚠ Satır 16 hatası: list indices must be integers or slices, not str

⚠ Satır 17 hatası: list indices must be integers or slices, not str

⚠ Satır 18 hatası: list indices must be integers or slices, not str

⚠ Satır 19 hatası: list indices must b

In [ ]:
import pandas as pd

checkpoint = pd.read_csv(f"{CHECKPOINT_PATH}generation_qwen25_7b_{TODAY}.csv")

print("=" * 70)
print("CHECKPOINT İNCELEMESİ")
print("=" * 70)
print(f"Toplam satır: {len(checkpoint)}")
print(f"Kolonlar: {list(checkpoint.columns)}")

print("\n" + "=" * 70)
print("İLK SATIRIN HAM MODEL CEVABI (model_raw_response)")
print("=" * 70)
raw = checkpoint['model_raw_response'].iloc[0]
print(f"Tip: {type(raw)}")
print(f"Uzunluk: {len(str(raw))} karakter")
print(f"İlk 1000 karakter:")
print(repr(raw[:1000]) if isinstance(raw, str) else raw)

print("\n" + "=" * 70)
print("PARSE EDİLMİŞ FIELDLER (boş mu dolu mu?)")
print("=" * 70)
print(f"model_reasoning (ilk satır): '{checkpoint['model_reasoning'].iloc[0]}'")
print(f"model_answer (ilk satır): '{checkpoint['model_answer'].iloc[0]}'")
print(f"model_confidence (ilk satır): {checkpoint['model_confidence'].iloc[0]}")
print(f"error (ilk satır): '{checkpoint['error'].iloc[0]}'")

print("\n" + "=" * 70)
print("BOŞ OLMAYAN model_reasoning SAYISI")
print("=" * 70)
non_empty = checkpoint['model_reasoning'].astype(str).str.strip().str.len() > 0
print(f"Dolu model_reasoning: {non_empty.sum()} / {len(checkpoint)}")

print("\n" + "=" * 70)
print("3 FARKLI SATIRIN HAM CEVABI")
print("=" * 70)
for i in [0, 100, 400]:
    print(f"\n--- Satır {i} ---")
    raw = str(checkpoint['model_raw_response'].iloc[i])
    print(raw[:500])
    print("...")

CHECKPOINT İNCELEMESİ
Toplam satır: 489
Kolonlar: ['question_id', 'category', 'difficulty', 'context', 'question', 'gold_evidence', 'gold_answer', 'gold_evidence_list', 'evidence_count', 'model_raw_response', 'model_reasoning', 'model_answer', 'model_confidence', 'error']

İLK SATIRIN HAM MODEL CEVABI (model_raw_response)
Tip: <class 'str'>
Uzunluk: 307 karakter
İlk 1000 karakter:
'{\n  "reasoning": "Sevda toplama sırasında, yüzler basamağı 4 artırılıyor. Bu durum, C sayısının yüzler basamağı 4 arttığı anlamına geliyor. D sayıları, C sayısının yüzler basamağını 4 artırarak elde edilen sonucu oluşturuyor. Bu nedenle, D sayısının değeri 4 olmalı.",\n  "answer": "4",\n  "confidence": 100\n}'

PARSE EDİLMİŞ FIELDLER (boş mu dolu mu?)
model_reasoning (ilk satır): 'Sevda toplama sırasında, yüzler basamağı 4 artırılıyor. Bu durum, C sayısının yüzler basamağı 4 arttığı anlamına geliyor. D sayıları, C sayısının yüzler basamağını 4 artırarak elde edilen sonucu oluşturuyor. Bu nedenle, D sayısını

In [ ]:
import inspect

print("=" * 70)
print("FAITHFULNESS_UTILS FONKSİYON İMZALARI")
print("=" * 70)

funcs_to_check = [
    'calculate_semantic_similarity',
    'calculate_nli_score',
    'calculate_faithfulness',
    'calculate_accuracy',
    'calculate_soft_accuracy',
    'calculate_epistemic_metrics',
]

for fname in funcs_to_check:
    try:
        from core.faithfulness_utils import __dict__ as fu_dict
        func = fu_dict.get(fname)
        if func is None:
            # Direkt module'den almayı dene
            import core.faithfulness_utils as fu
            func = getattr(fu, fname, None)

        if func:
            sig = inspect.signature(func)
            print(f"\n{fname}{sig}")
            # Docstring varsa göster
            if func.__doc__:
                print(f"  Docstring: {func.__doc__[:200]}")
        else:
            print(f"\n{fname}: BULUNAMADI")
    except Exception as e:
        print(f"\n{fname}: HATA - {e}")

# Tek bir satırla deneme yapalım, hata tam olarak nerede çıkıyor görelim
print("\n" + "=" * 70)
print("ADIM ADIM TEST")
print("=" * 70)

import pandas as pd
checkpoint = pd.read_csv(f"{CHECKPOINT_PATH}generation_qwen25_7b_{TODAY}.csv")
row = checkpoint.iloc[0]

reasoning = str(row['model_reasoning'])
evidence = str(row['gold_evidence'])
answer = str(row['model_answer'])
gold = str(row['gold_answer'])

print(f"reasoning: {reasoning[:100]}...")
print(f"evidence: {evidence[:100]}...")
print(f"answer: {answer}")
print(f"gold: {gold}")

# Adım adım dene
import core.faithfulness_utils as fu

print("\n--- Adım 1: semantic similarity ---")
try:
    sem = fu.calculate_semantic_similarity(reasoning, evidence)
    print(f"✓ sem_sim = {sem}")
except Exception as e:
    print(f"✗ HATA: {e}")

print("\n--- Adım 2: NLI score ---")
try:
    nli = fu.calculate_nli_score(reasoning, evidence)
    print(f"✓ nli = {nli}")
except Exception as e:
    print(f"✗ HATA: {e}")

print("\n--- Adım 3: faithfulness ---")
try:
    faith = fu.calculate_faithfulness(sem, nli)
    print(f"✓ faith = {faith}")
except Exception as e:
    print(f"✗ HATA: {e}")

print("\n--- Adım 4: exact accuracy ---")
try:
    exact = fu.calculate_accuracy(answer, gold)
    print(f"✓ exact = {exact}")
except Exception as e:
    print(f"✗ HATA: {e}")

print("\n--- Adım 5: soft accuracy ---")
try:
    soft = fu.calculate_soft_accuracy(answer, gold)
    print(f"✓ soft = {soft}")
    print(f"  Tip: {type(soft)}")
except Exception as e:
    print(f"✗ HATA: {e}")
    # Belki 3 argüman istiyor
    try:
        soft = fu.calculate_soft_accuracy(answer, gold, None)
        print(f"  3 argümanla: {soft}")
    except Exception as e2:
        print(f"  3 argümanla da hata: {e2}")

print("\n--- Adım 6: epistemic metrics ---")
try:
    conf_norm = 1.0  # 100/100
    epi = fu.calculate_epistemic_metrics(conf_norm, 0.7)
    print(f"✓ epi = {epi}")
    print(f"  Tip: {type(epi)}")
except Exception as e:
    print(f"✗ HATA: {e}")

FAITHFULNESS_UTILS FONKSİYON İMZALARI

calculate_semantic_similarity(text1, text2)

calculate_nli_score(premise, hypothesis)

calculate_faithfulness(reasoning, evidence_list)

calculate_accuracy(model_answer, gold_answer)

calculate_soft_accuracy(model_answer, gold_answer, semantic_threshold=0.85, nli_threshold=0.4)
  Docstring: 
    Hibrit accuracy hesabı:
    1. Exact match varsa -> 1
    2. Semantic similarity >= threshold VE NLI entailment >= threshold -> 1
    3. Aksi halde -> 0
    

calculate_epistemic_metrics(confidence, faithfulness, accuracy)

ADIM ADIM TEST
reasoning: Sevda toplama sırasında, yüzler basamağı 4 artırılıyor. Bu durum, C sayısının yüzler basamağı 4 artt...
evidence: Sevda C sayısı ile D sayısını toplamak istiyor. | Sonucun C sayısının yüzler basamağını 4 artırarak ...
answer: 4
gold: 400

--- Adım 1: semantic similarity ---
✓ sem_sim = 0.9554380178451538

--- Adım 2: NLI score ---
✗ HATA: list indices must be integers or slices, not str

--- Adım 3: faithfulnes

In [ ]:
# ============================================================
# NLI PIPELINE'INI ESKİ KOD UYGUN ŞEKİLDE YENİDEN YÜKLE
# ============================================================
import gc
import torch
from transformers import pipeline
from sentence_transformers import SentenceTransformer

# Önce eski global modelleri temizle
try:
    del nli_pipe
    del sem_model
    gc.collect()
    torch.cuda.empty_cache()
    print("✓ Eski modeller temizlendi")
except:
    pass

print("► Sentence-Transformers yükleniyor...")
sem_model = SentenceTransformer('intfloat/multilingual-e5-large')

print("► NLI pipeline yükleniyor (TOP_K YOK, tek result döndürür)...")
# Eski faithfulness_utils'in beklediği şekilde — top_k YOK
nli_pipe = pipeline(
    "text-classification",
    model="joeddav/xlm-roberta-large-xnli",
    device=0 if torch.cuda.is_available() else -1
    # top_k parametresi YOK — pipeline'ın varsayılan davranışı
)

# Global modelleri faithfulness_utils'e tanıt
from core.faithfulness_utils import set_models
set_models(sem_model, nli_pipe)

print("✓ Modeller hazır ve set_models ile tanıtıldı")

# Hemen test edelim
print("\n--- Test ---")
import core.faithfulness_utils as fu
try:
    nli = fu.calculate_nli_score(
        "Sevda yüzler basamağını 4 artırdı.",
        "Sevda C sayısının yüzler basamağını 4 artırarak sonuca ulaşıyor."
    )
    print(f"✓ NLI score test başarılı: {nli}")
except Exception as e:
    print(f"✗ Hala hata: {e}")

✓ Eski modeller temizlendi
► Sentence-Transformers yükleniyor...


Loading weights:   0%|          | 0/391 [00:00<?, ?it/s]

XLMRobertaModel LOAD REPORT from: intfloat/multilingual-e5-large
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


► NLI pipeline yükleniyor (TOP_K YOK, tek result döndürür)...


Loading weights:   0%|          | 0/393 [00:00<?, ?it/s]

XLMRobertaForSequenceClassification LOAD REPORT from: joeddav/xlm-roberta-large-xnli
Key                             | Status     |  | 
--------------------------------+------------+--+-
roberta.pooler.dense.bias       | UNEXPECTED |  | 
roberta.embeddings.position_ids | UNEXPECTED |  | 
roberta.pooler.dense.weight     | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✓ Modeller hazır ve set_models ile tanıtıldı

--- Test ---
✗ Hala hata: list indices must be integers or slices, not str


---
## 🔧 BÖLÜM D — Trendyol LLM 7B Chat Test

**Model:** Trendyol/Trendyol-LLM-7b-chat-v1.0
**Tahmini süre:** ~1-1.5 saat
**Özellik:** Türkçe-spesifik fine-tune
---

In [ ]:
# ============================================================
# D: Trendyol LLM 7B Chat
# ============================================================

MODEL_ID = "Trendyol/Trendyol-LLM-7b-chat-v1.0"
MODEL_SHORT_NAME = "trendyol_7b"
MODEL_DISPLAY_NAME = "Trendyol LLM 7B"

model, tokenizer = load_model_4bit(MODEL_ID)
gen_df = run_model_on_dataset(model, tokenizer, df, MODEL_SHORT_NAME)
cleanup_model(model, tokenizer)

scored_df = score_dataset(gen_df)

output_path = f"{RESULTS_PATH}evaluation_acik_{MODEL_SHORT_NAME}_{TODAY}.csv"
scored_df.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f"\n✓ Kaydedildi: {output_path}")

print_summary(scored_df, MODEL_DISPLAY_NAME)

---
## 🔧 BÖLÜM E — Llama 3.1 8B Instruct Test

**Model:** meta-llama/Llama-3.1-8B-Instruct
**Tahmini süre:** ~1.5-2 saat
**Not:** HuggingFace'de erişim onayı gerekiyor (zaten alındı)
---

In [ ]:
# ============================================================
# E: Llama 3.1 8B Instruct
# ============================================================

MODEL_ID = "meta-llama/Llama-3.1-8B-Instruct"
MODEL_SHORT_NAME = "llama31_8b"
MODEL_DISPLAY_NAME = "Llama 3.1 8B Instruct"

model, tokenizer = load_model_4bit(MODEL_ID)
gen_df = run_model_on_dataset(model, tokenizer, df, MODEL_SHORT_NAME)
cleanup_model(model, tokenizer)

scored_df = score_dataset(gen_df)

output_path = f"{RESULTS_PATH}evaluation_acik_{MODEL_SHORT_NAME}_{TODAY}.csv"
scored_df.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f"\n✓ Kaydedildi: {output_path}")

print_summary(scored_df, MODEL_DISPLAY_NAME)

---
## 🔧 BÖLÜM F — Gemma 2 9B IT Test

**Model:** google/gemma-2-9b-it
**Tahmini süre:** ~1.5-2 saat
**GPU bellek:** ~8-9 GB (4-bit)
---

In [ ]:
# ============================================================
# F: Gemma 2 9B IT
# ============================================================

MODEL_ID = "google/gemma-2-9b-it"
MODEL_SHORT_NAME = "gemma2_9b"
MODEL_DISPLAY_NAME = "Gemma 2 9B IT"

model, tokenizer = load_model_4bit(MODEL_ID)
gen_df = run_model_on_dataset(model, tokenizer, df, MODEL_SHORT_NAME)
cleanup_model(model, tokenizer)

scored_df = score_dataset(gen_df)

output_path = f"{RESULTS_PATH}evaluation_acik_{MODEL_SHORT_NAME}_{TODAY}.csv"
scored_df.to_csv(output_path, index=False, encoding='utf-8-sig')
print(f"\n✓ Kaydedildi: {output_path}")

print_summary(scored_df, MODEL_DISPLAY_NAME)